In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import pickle
from pathlib import Path

Path("data").mkdir(exist_ok=True)

periods = ["2011_2015", "2016_2020", "2021_2025"]

In [ ]:
authors_list = []

for period in periods:
    df = pd.read_csv(f"data/authors_{period}.csv")
    df["period"] = period
    authors_list.append(df)

authors_15 = pd.concat(authors_list, ignore_index=True)
authors_15 = authors_15.drop_duplicates(subset=["paper_id", "author_id"])

print("authors rows:", len(authors_15))
print("unique papers:", authors_15["paper_id"].nunique())
print("unique authors:", authors_15["author_id"].nunique())

authors rows: 1225426
unique papers: 469354
unique authors: 556581


In [ ]:
pairs_list = []

for period in periods:
    df = pd.read_csv(f"data/pairs_{period}.csv")
    df["period"] = period
    pairs_list.append(df)

pairs_15 = pd.concat(pairs_list, ignore_index=True)

if "publication_year" in pairs_15.columns:
    pairs_15 = pairs_15.drop(columns=["publication_year"])

paper_year = authors_15[["paper_id", "publication_year"]].drop_duplicates()
pairs_15 = pairs_15.merge(paper_year, on="paper_id", how="left")

pairs_15 = pairs_15.dropna(subset=["author_1_id", "author_2_id"])
pairs_15["u"] = pairs_15[["author_1_id", "author_2_id"]].min(axis=1)
pairs_15["v"] = pairs_15[["author_1_id", "author_2_id"]].max(axis=1)

pairs_15 = pairs_15.drop_duplicates(subset=["paper_id", "u", "v"])

print("pairs rows:", len(pairs_15))
print("unique papers in pairs:", pairs_15["paper_id"].nunique())

pairs rows: 1879929
unique papers in pairs: 315611


In [ ]:
pairs_15.to_csv("data/pairs_2011_2025.csv", index=False)

In [ ]:
paper_sizes = (authors_15.groupby("paper_id")["author_id"].nunique().rename("num_authors").reset_index())

papers_15 = (authors_15[["paper_id", "paper_title", "publication_year"]].drop_duplicates("paper_id").merge(paper_sizes, on="paper_id", how="left"))

title = papers_15["paper_title"].fillna("").str.lower().str.strip()

bad_title = title.str.match(r"^(table of contents|contents|front matter|back matter|editorial board|masthead|issue information|volume contents)\b")

bad_papers = papers_15[bad_title].copy()
bad_papers["num_pairs"] = bad_papers["num_authors"] * (bad_papers["num_authors"] - 1) / 2

print("bad papers:", len(bad_papers))
print("authors in bad papers:", bad_papers["num_authors"].sum())
print("possible false pairs:", int(bad_papers["num_pairs"].sum()))

bad_papers.sort_values("num_authors", ascending=False).head(20)

bad papers: 344
authors in bad papers: 1352
possible false pairs: 4038


,paper_id,paper_title,publication_year,num_authors,num_pairs
231648,https://openalex.org/W4213024521,Table of Contents,2019,15,105.0
299302,https://openalex.org/W4252296935,Table of Contents,2018,15,105.0
409425,https://openalex.org/W4245009974,Table of Contents,2021,15,105.0
299083,https://openalex.org/W4250272453,Table of Contents,2020,15,105.0
415446,https://openalex.org/W4311080786,Table of Contents,2022,14,91.0
298373,https://openalex.org/W4243173682,Table of Contents,2020,14,91.0
298410,https://openalex.org/W4243587731,Table of Contents,2020,14,91.0
429715,https://openalex.org/W4389249266,Table of Contents,2023,14,91.0
298369,https://openalex.org/W4243142224,Table of Contents,2020,14,91.0
409478,https://openalex.org/W4246613126,Table of Contents,2021,13,78.0


In [ ]:
bad_paper_ids = set(bad_papers["paper_id"])

authors_15 = authors_15[~authors_15["paper_id"].isin(bad_paper_ids)].copy()
pairs_15 = pairs_15[~pairs_15["paper_id"].isin(bad_paper_ids)].copy()

print("authors rows after cleaning:", len(authors_15))
print("pairs rows after cleaning:", len(pairs_15))
print("unique papers after cleaning:", authors_15["paper_id"].nunique())
print("unique papers in pairs after cleaning:", pairs_15["paper_id"].nunique())

authors rows after cleaning: 1224074
pairs rows after cleaning: 1875891
unique papers after cleaning: 469010
unique papers in pairs after cleaning: 315386


In [ ]:
bad_papers.to_csv("data/bad_paratext_papers_2011_2025.csv", index=False)
authors_15.to_csv("data/authors_2011_2025_clean.csv", index=False)
pairs_15.to_csv("data/pairs_2011_2025_clean.csv", index=False)

In [ ]:
pairs_focal_15 = pairs_15[(pairs_15["author_1_in_target"] == 1) | (pairs_15["author_2_in_target"] == 1)].copy()
print("focal pairs:", len(pairs_focal_15))
print("unique papers in focal pairs:", pairs_focal_15["paper_id"].nunique())

pairs_focal_15.to_csv("data/pairs_focal_2011_2025.csv", index=False)

focal pairs: 1552322
unique papers in focal pairs: 314833


In [ ]:
edges_15 = (
    pairs_focal_15
    .groupby(["u", "v"], as_index=False)
    .agg(weight=("paper_id", "nunique"), first_year=("publication_year", "min"),
         last_year=("publication_year", "max"),
         years=("publication_year", lambda x: ";".join(map(str, sorted(x.dropna().astype(int).unique())))),
         periods=("period", lambda x: ";".join(sorted(x.dropna().unique())))).rename(columns={"u": "author_1_id", "v": "author_2_id"}))



print("edges:", len(edges_15))
print("sum weight:", edges_15["weight"].sum())
print("max weight:", edges_15["weight"].max())

edges_15.to_csv("data/edges_2011_2025.csv", index=False)

edges: 1204212
sum weight: 1552322
max weight: 166


In [ ]:
edges_15 = edges_15[edges_15["weight"] <= 50].copy()
print("edges after filter:", len(edges_15))
print("max weight after filter:", edges_15["weight"].max())

edges_15.to_csv("data/edges_2011_2025.csv", index=False)

edges after filter: 1204181
max weight after filter: 50


In [ ]:
node_rows = []

for author_id, group in authors_15.groupby("author_id"):
    author_name = group["author_name"].dropna().iloc[0] if group["author_name"].notna().any() else pd.NA
    num_papers = group["paper_id"].nunique()
    in_target_countries = int(group["in_target_countries"].max())

    all_country_counts = {}
    target_country_counts = {}

    for x in group["author_countries_all"].dropna():
        for c in str(x).split(";"):
            if c != "":
                all_country_counts[c] = all_country_counts.get(c, 0) + 1

    for x in group["author_countries_target"].dropna():
        for c in str(x).split(";"):
            if c != "":
                target_country_counts[c] = target_country_counts.get(c, 0) + 1

    all_countries = sorted(all_country_counts.keys())
    target_countries = sorted(target_country_counts.keys())

    main_country_all = max(all_country_counts, key=all_country_counts.get) if all_country_counts else pd.NA
    main_country_target = max(target_country_counts, key=target_country_counts.get) if target_country_counts else pd.NA

    node_rows.append({
        "author_id": author_id,
        "author_name": author_name,
        "author_countries_all": ";".join(all_countries),
        "author_countries_target": ";".join(target_countries),
        "in_target_countries": in_target_countries,
        "main_country_all": main_country_all,
        "main_country_target": main_country_target,
        "num_papers": num_papers})

nodes_15 = pd.DataFrame(node_rows)

print("nodes:", len(nodes_15))
print(nodes_15["main_country_target"].value_counts(dropna=False))

nodes: 555415
main_country_target
us      230260
<NA>    142065
gb       64744
ru       42545
br       35902
it       26434
kr       13465
Name: count, dtype: int64


In [ ]:
nodes_15[nodes_15["main_country_target"].isna()]["main_country_all"].value_counts().head(30)

,count
main_country_all,
cn,17267
de,5753
ca,5137
fr,4324
au,4306
in,3714
nl,3626
es,3539
ch,2593


In [ ]:
gender_list = []

for period in periods:
    df = pd.read_csv(f"data/nodes_{period}_with_gender.csv")
    df = df[["author_id", "name_inferred_gender"]].copy()
    df["period"] = period
    gender_list.append(df)

gender_15 = pd.concat(gender_list, ignore_index=True)
gender_15["name_inferred_gender"] = gender_15["name_inferred_gender"].fillna("unknown")

gender_known = gender_15[gender_15["name_inferred_gender"].isin(["female", "male"])].copy()

gender_agg = (gender_known.groupby("author_id")["name_inferred_gender"].agg(lambda x: x.value_counts().index[0]).reset_index())

nodes_15 = nodes_15.merge(gender_agg, on="author_id", how="left")
nodes_15["name_inferred_gender"] = nodes_15["name_inferred_gender"].fillna("unknown")
nodes_15["gender"] = nodes_15["name_inferred_gender"]

print(nodes_15["name_inferred_gender"].value_counts(dropna=False))

name_inferred_gender
male       252597
female     164973
unknown    137845
Name: count, dtype: int64


In [ ]:
g_15 = nx.Graph()

for _, row in edges_15.iterrows():
    g_15.add_edge(
        row["author_1_id"],
        row["author_2_id"],
        weight=int(row["weight"]),
        first_year=int(row["first_year"]),
        last_year=int(row["last_year"]),
        years=row["years"],
        periods=row["periods"])

g_15.add_nodes_from(nodes_15["author_id"])

node_attrs = nodes_15.set_index("author_id")[[
    "author_name",
    "author_countries_all",
    "author_countries_target",
    "in_target_countries",
    "main_country_all",
    "main_country_target",
    "num_papers",
    "name_inferred_gender",
    "gender"]].to_dict("index")

nx.set_node_attributes(g_15, node_attrs)

print("nodes:", g_15.number_of_nodes())
print("edges:", g_15.number_of_edges())
print("isolates:", nx.number_of_isolates(g_15))

nodes: 555415
edges: 1204181
isolates: 65470


In [ ]:
degree_dict = dict(g_15.degree())
weighted_degree_dict = dict(g_15.degree(weight="weight"))

nodes_15["degree"] = nodes_15["author_id"].map(degree_dict).fillna(0).astype(int)
nodes_15["weighted_degree"] = nodes_15["author_id"].map(weighted_degree_dict).fillna(0).astype(int)

largest_component = max(nx.connected_components(g_15), key=len)
largest_component_set = set(largest_component)

nodes_15["is_in_giant_component"] = nodes_15["author_id"].isin(largest_component_set)

print(nodes_15[["num_papers", "degree", "weighted_degree"]].describe())
print(nodes_15["is_in_giant_component"].value_counts())

          num_papers         degree  weighted_degree
count  555415.000000  555415.000000    555415.000000
mean        2.203891       4.336149         5.582197
std         4.465764       7.124167        12.692703
min         1.000000       0.000000         0.000000
25%         1.000000       1.000000         1.000000
50%         1.000000       2.000000         3.000000
75%         2.000000       5.000000         6.000000
max       399.000000     360.000000       955.000000
is_in_giant_component
True     309379
False    246036
Name: count, dtype: int64


In [ ]:
nodes_15.to_csv("data/nodes_2011_2025_with_gender_metrics.csv", index=False)

with open("data/g_2011_2025.pkl", "wb") as f:
    pickle.dump(g_15, f)

print("saved")

saved


In [ ]:
edge_gender_15 = edges_15.copy()

lookup = nodes_15[["author_id", "gender", "main_country_target"]].copy()

edge_gender_15 = edge_gender_15.merge(
    lookup.rename(columns={
        "author_id": "author_1_id",
        "gender": "gender_1",
        "main_country_target": "country_1"}),
    on="author_1_id",
    how="left")

edge_gender_15 = edge_gender_15.merge(
    lookup.rename(columns={
        "author_id": "author_2_id",
        "gender": "gender_2",
        "main_country_target": "country_2"}),
    on="author_2_id",
    how="left")

In [ ]:
def edge_gender(g1, g2):
    if pd.isna(g1) or pd.isna(g2):
        return "unknown"
    if g1 == "unknown" or g2 == "unknown":
        return "unknown"
    if g1 == "female" and g2 == "female":
        return "female-female"
    if g1 == "male" and g2 == "male":
        return "male-male"
    if {g1, g2} == {"female", "male"}:
        return "female-male"
    return "unknown"

In [ ]:
edge_gender_15["edge_gender_type"] = edge_gender_15.apply(lambda row: edge_gender(row["gender_1"], row["gender_2"]), axis=1)

edge_gender_15.to_csv("data/edges_2011_2025_with_gender.csv", index=False)

edge_gender_15["edge_gender_type"].value_counts()

,count
edge_gender_type,
unknown,366111
female-male,343686
male-male,337058
female-female,157326


METRICS

In [ ]:
Path("data/summaries").mkdir(exist_ok=True)

In [ ]:
papers_by_year = (authors_15.groupby("publication_year").agg(
    papers=("paper_id", "nunique"),
    author_paper_rows=("author_id", "size"),
    authors=("author_id", "nunique")).reset_index())

pairs_by_year = (pairs_15.groupby("publication_year").agg(
    pairs=("paper_id", "size"),
    papers_with_pairs=("paper_id", "nunique")).reset_index())

focal_pairs_by_year = (pairs_focal_15.groupby("publication_year").agg(
    focal_pairs=("paper_id", "size"),
    focal_papers_with_pairs=("paper_id", "nunique")).reset_index())

edges_by_year = (pairs_focal_15
                 .groupby(["publication_year", "u", "v"], as_index=False)
                 .agg(weight=("paper_id", "nunique"))
                 .groupby("publication_year")
                 .agg(edges=("weight", "size"),
                      total_weight=("weight", "sum"),
                      max_weight=("weight", "max")).reset_index())

summary_year = (papers_by_year
                .merge(pairs_by_year, on="publication_year", how="left")
                .merge(focal_pairs_by_year, on="publication_year", how="left")
                .merge(edges_by_year, on="publication_year", how="left"))

summary_year.to_csv("data/summaries/summary_year_2011_2025.csv", index=False)

summary_year

,publication_year,papers,author_paper_rows,authors,pairs,papers_with_pairs,focal_pairs,focal_papers_with_pairs,edges,total_weight,max_weight
0,2011,27634,59344,42127,63290,16684,53692,16673,48203,53692,10
1,2012,28719,63926,45639,73930,17623,62774,17597,55717,62774,19
2,2013,27988,64340,47088,81335,17319,68986,17304,61549,68986,17
3,2014,29128,67595,49914,85409,18163,72444,18129,65806,72444,13
4,2015,30081,71098,52836,92250,18986,78050,18956,70852,78050,15
5,2016,29236,70933,53168,97124,18852,82229,18840,74864,82229,19
6,2017,30030,75437,56228,107179,20009,91351,19990,82647,91351,17
7,2018,31078,81579,61303,125352,21114,105497,21071,95159,105497,19
8,2019,32084,85263,65206,132964,21836,112199,21810,101845,112199,23
9,2020,36738,100426,74380,161473,25673,133978,25654,119830,133978,33


In [ ]:
def author_summary(df, group_cols):
    out = (df.groupby(group_cols).agg(
            authors=("author_id", "nunique"),
            female_authors=("gender", lambda x: (x == "female").sum()),
            male_authors=("gender", lambda x: (x == "male").sum()),
            unknown_authors=("gender", lambda x: (x == "unknown").sum()),
            papers_mean=("num_papers", "mean"),
            papers_median=("num_papers", "median"),
            papers_std=("num_papers", "std"),
            papers_p75=("num_papers", lambda x: x.quantile(0.75)),
            papers_p90=("num_papers", lambda x: x.quantile(0.90)),
            papers_p95=("num_papers", lambda x: x.quantile(0.95)),
            degree_mean=("degree", "mean"),
            degree_median=("degree", "median"),
            degree_std=("degree", "std"),
            degree_p75=("degree", lambda x: x.quantile(0.75)),
            degree_p90=("degree", lambda x: x.quantile(0.90)),
            degree_p95=("degree", lambda x: x.quantile(0.95)),
            weighted_degree_mean=("weighted_degree", "mean"),
            weighted_degree_median=("weighted_degree", "median"),
            weighted_degree_std=("weighted_degree", "std"),
            weighted_degree_p75=("weighted_degree", lambda x: x.quantile(0.75)),
            weighted_degree_p90=("weighted_degree", lambda x: x.quantile(0.90)),
            weighted_degree_p95=("weighted_degree", lambda x: x.quantile(0.95)),
            giant_component_share=("is_in_giant_component", "mean"))
        .reset_index())

    out["female_share"] = out["female_authors"] / out["authors"]
    out["male_share"] = out["male_authors"] / out["authors"]
    out["unknown_share"] = out["unknown_authors"] / out["authors"]
    out["classified_authors"] = out["female_authors"] + out["male_authors"]
    out["female_share_classified"] = out["female_authors"] / out["classified_authors"]
    out["male_share_classified"] = out["male_authors"] / out["classified_authors"]

    return out.round(4)

In [ ]:
nodes_target_15 = nodes_15[nodes_15["main_country_target"].notna()].copy()

summary_authors_country = author_summary(nodes_target_15, ["main_country_target"])

summary_authors_country.to_csv("data/summaries/summary_authors_country_2011_2025.csv", index=False)

summary_authors_country

,main_country_target,authors,female_authors,male_authors,unknown_authors,papers_mean,papers_median,papers_std,papers_p75,papers_p90,...,weighted_degree_p75,weighted_degree_p90,weighted_degree_p95,giant_component_share,female_share,male_share,unknown_share,classified_authors,female_share_classified,male_share_classified
0,br,35902,11884,19590,4428,1.8264,1.0,2.9645,2.0,3.0,...,5.0,9.0,14.0,0.4213,0.3310,0.5457,0.1233,31474,0.3776,0.6224
1,gb,64744,21454,34703,8587,2.7427,1.0,5.8107,2.0,5.0,...,8.0,15.0,25.0,0.5662,0.3314,0.5360,0.1326,56157,0.3820,0.6180
2,it,26434,9506,14883,2045,3.0441,1.0,5.7740,2.0,7.0,...,9.0,17.0,27.0,0.5802,0.3596,0.5630,0.0774,24389,0.3898,0.6102
3,kr,13465,1962,2841,8662,2.3577,1.0,5.1080,2.0,4.0,...,6.0,11.0,16.0,0.5747,0.1457,0.2110,0.6433,4803,0.4085,0.5915
4,ru,42545,13509,12359,16677,1.6814,1.0,2.8511,1.0,3.0,...,3.0,6.0,9.0,0.2362,0.3175,0.2905,0.3920,25868,0.5222,0.4778
5,us,230260,79002,119519,31739,2.5771,1.0,5.3107,2.0,5.0,...,8.0,16.0,25.0,0.5817,0.3431,0.5191,0.1378,198521,0.3980,0.6020


In [ ]:
nodes_target = nodes_15[nodes_15["main_country_target"].notna()].copy()

if "gender" not in nodes_target.columns:
    nodes_target["gender"] = nodes_target["name_inferred_gender"]

In [ ]:
author_stats_country = (nodes_target.groupby("main_country_target").agg(
        num_authors=("author_id", "nunique"),

        papers_mean=("num_papers", "mean"),
        papers_median=("num_papers", "median"),
        papers_std=("num_papers", "std"),
        papers_p90=("num_papers", lambda x: x.quantile(0.90)),

        degree_mean=("degree", "mean"),
        degree_median=("degree", "median"),
        degree_std=("degree", "std"),
        degree_p90=("degree", lambda x: x.quantile(0.90)),

        weighted_degree_mean=("weighted_degree", "mean"),
        weighted_degree_median=("weighted_degree", "median"),
        weighted_degree_std=("weighted_degree", "std"),
        weighted_degree_p90=("weighted_degree", lambda x: x.quantile(0.90)),

        giant_component_share=("is_in_giant_component", "mean")).round(3))

author_stats_country

,num_authors,papers_mean,papers_median,papers_std,papers_p90,degree_mean,degree_median,degree_std,degree_p90,weighted_degree_mean,weighted_degree_median,weighted_degree_std,weighted_degree_p90,giant_component_share
main_country_target,,,,,,,,,,,,,,
br,35902,1.826,1.0,2.965,3.0,3.754,3.0,4.636,8.0,4.423,3.0,7.106,9.0,0.421
gb,64744,2.743,1.0,5.811,5.0,5.456,3.0,8.902,12.0,7.192,3.0,15.478,15.0,0.566
it,26434,3.044,1.0,5.774,7.0,5.537,4.0,7.301,13.0,7.748,4.0,13.606,17.0,0.580
kr,13465,2.358,1.0,5.108,4.0,4.096,3.0,5.478,9.0,5.203,3.0,10.459,11.0,0.575
ru,42545,1.681,1.0,2.851,3.0,2.436,2.0,3.551,5.0,2.861,2.0,5.996,6.0,0.236
us,230260,2.577,1.0,5.311,5.0,5.715,3.0,8.808,13.0,7.522,4.0,16.143,16.0,0.582


In [ ]:
author_stats_country.to_csv("data/author_stats_country_2011_2025.csv")

In [ ]:
gender_counts_country = pd.crosstab(nodes_target["main_country_target"], nodes_target["gender"])

gender_counts_country

gender,female,male,unknown
main_country_target,,,
br,11884,19590,4428
gb,21454,34703,8587
it,9506,14883,2045
kr,1962,2841,8662
ru,13509,12359,16677
us,79002,119519,31739


In [ ]:
gender_shares_country = pd.crosstab(nodes_target["main_country_target"], nodes_target["gender"], normalize="index").round(3)

gender_shares_country

gender,female,male,unknown
main_country_target,,,
br,0.331,0.546,0.123
gb,0.331,0.536,0.133
it,0.360,0.563,0.077
kr,0.146,0.211,0.643
ru,0.318,0.290,0.392
us,0.343,0.519,0.138


In [ ]:
gender_counts_country.to_csv("data/gender_counts_country_2011_2025.csv")
gender_shares_country.to_csv("data/gender_shares_country_2011_2025.csv")

In [ ]:
author_stats_country_gender = (nodes_target[nodes_target["gender"].isin(["female", "male"])]
                               .groupby(["main_country_target", "gender"]).agg(
                                   num_authors=("author_id", "nunique"),

                                   papers_mean=("num_papers", "mean"),
                                   papers_median=("num_papers", "median"),
                                   papers_std=("num_papers", "std"),

                                   degree_mean=("degree", "mean"),
                                   degree_median=("degree", "median"),
                                   degree_std=("degree", "std"),

                                   weighted_degree_mean=("weighted_degree", "mean"),
                                   weighted_degree_median=("weighted_degree", "median"),
                                   weighted_degree_std=("weighted_degree", "std")).round(3))

author_stats_country_gender

num_authors  papers_mean  papers_median  \
main_country_target gender                                            
br                  female        11884        1.514            1.0   
                    male          19590        2.057            1.0   
gb                  female        21454        2.184            1.0   
                    male          34703        3.245            1.0   
it                  female         9506        2.524            1.0   
                    male          14883        3.579            1.0   
kr                  female         1962        1.989            1.0   
                    male           2841        2.520            1.0   
ru                  female        13509        1.610            1.0   
                    male          12359        1.933            1.0   
us                  female        79002        2.100            1.0   
                    male         119519        2.990            1.0   

                            papers_std  degree_mean  degree_median  \
main_country_target gender                                           
br                  female       1.911        3.723            3.0   
                    male         3.565        3.740            2.0   
gb                  female       3.675        5.413            3.0   
                    male         7.033        5.656            3.0   
it                  female       4.117        5.175            3.0   
                    male         6.858        5.778            3.0   
kr                  female       3.266        3.916            3.0   
                    male         5.703        4.153            3.0   
ru                  female       2.026        2.535            2.0   
                    male         4.153        2.521            2.0   
us                  female       3.657        5.894            4.0   
                    male         6.238        5.797            3.0   

                            degree_std  weighted_degree_mean  \
main_country_target gender                                     
br                  female       4.121                 4.205   
                    male         4.918                 4.543   
gb                  female       8.057                 6.684   
                    male         9.840                 7.839   
it                  female       6.310                 6.920   
                    male         8.112                 8.490   
kr                  female       4.775                 4.675   
                    male         6.041                 5.333   
ru                  female       2.996                 2.883   
                    male         4.624                 3.141   
us                  female       8.255                 7.455   
                    male         9.462                 7.897   

                            weighted_degree_median  weighted_degree_std  
main_country_target gender                                               
br                  female                     3.0                5.759  
                    male                       3.0                7.892  
gb                  female                     4.0               12.701  
                    male                       3.0               17.781  
it                  female                     4.0               11.036  
                    male                       4.0               15.601  
kr                  female                     3.0                6.970  
                    male                       3.0               11.680  
ru                  female                     2.0                4.355  
                    male                       2.0                8.514  
us                  female                     4.0               14.929  
                    male                       4.0               17.477

In [ ]:
author_stats_country_gender.to_csv("data/author_stats_country_gender_2011_2025.csv")

In [ ]:
within_edges = edge_gender_15[
    edge_gender_15["country_1"].notna() &
    edge_gender_15["country_2"].notna() &
    (edge_gender_15["country_1"] == edge_gender_15["country_2"]) &
    (edge_gender_15["edge_gender_type"] != "unknown")].copy()

within_edges["country"] = within_edges["country_1"]

In [ ]:
edge_gender_counts_country = pd.crosstab(
    within_edges["country"],
    within_edges["edge_gender_type"])

edge_gender_counts_country

edge_gender_type,female-female,female-male,male-male
country,,,
br,7926,17151,18275
gb,16758,37308,35074
it,7749,19272,18750
kr,556,1026,900
ru,6928,8278,4481
us,90434,180158,162624


In [ ]:
edge_gender_shares_country = pd.crosstab(
    within_edges["country"],
    within_edges["edge_gender_type"],
    normalize="index").round(3)

edge_gender_shares_country

edge_gender_type,female-female,female-male,male-male
country,,,
br,0.183,0.396,0.422
gb,0.188,0.419,0.393
it,0.169,0.421,0.410
kr,0.224,0.413,0.363
ru,0.352,0.420,0.228
us,0.209,0.416,0.375


In [ ]:
edge_gender_counts_country.to_csv("data/edge_gender_counts_country_2011_2025.csv")
edge_gender_shares_country.to_csv("data/edge_gender_shares_country_2011_2025.csv")

In [ ]:
country_assort_rows = []

for country in sorted(nodes_target["main_country_target"].unique()):
    country_nodes = [
        n for n, d in g_15.nodes(data=True)
        if d.get("main_country_target") == country and d.get("gender") in ["female", "male"]]

    subg = g_15.subgraph(country_nodes).copy()

    if subg.number_of_edges() == 0:
        assort = np.nan
    else:
        assort = nx.attribute_assortativity_coefficient(subg, "gender")

    country_assort_rows.append({
        "country": country,
        "num_nodes": subg.number_of_nodes(),
        "num_edges": subg.number_of_edges(),
        "gender_assortativity": assort})

country_assort_table = pd.DataFrame(country_assort_rows).round(3)

country_assort_table

,country,num_nodes,num_edges,gender_assortativity
0,br,31474,43352,0.161
1,gb,56157,89140,0.126
2,it,24389,45771,0.106
3,kr,4803,2482,0.157
4,ru,25868,19687,0.146
5,us,198521,433216,0.145


In [ ]:
country_assort_table.to_csv("data/country_assortativity_2011_2025.csv", index=False)

In [ ]:
year_rows = []

for year in range(2011, 2026):
    G = year_graph(year)
    H = classified_graph(G)

    if H.number_of_edges() == 0:
        continue

    genders = np.array([H.nodes[n]["gender"] for n in H.nodes()])
    female_share = (genders == "female").mean()

    assort = nx.attribute_assortativity_coefficient(H, "gender")

    same_obs = sum(
        1 for u, v in H.edges()
        if H.nodes[u]["gender"] == H.nodes[v]["gender"]) / H.number_of_edges()

    nodes = list(H.nodes())
    rng = np.random.default_rng(year)
    same_null = []

    for _ in range(100):
        shuffled = rng.permutation(genders)
        gender_rand = dict(zip(nodes, shuffled))

        same = sum(
            1 for u, v in H.edges()
            if gender_rand[u] == gender_rand[v]) / H.number_of_edges()

        same_null.append(same)

    same_null = np.array(same_null)

    year_rows.append({
        "year": year,
        "num_nodes": H.number_of_nodes(),
        "num_edges": H.number_of_edges(),
        "female_share_pct": 100 * female_share,
        "assortativity": assort,
        "same_gender_share_obs_pct": 100 * same_obs,
        "same_gender_share_null_pct": 100 * same_null.mean(),
        "same_gender_excess_pp": 100 * (same_obs - same_null.mean())})

yearly_bias_table = pd.DataFrame(year_rows).round(3)

yearly_bias_table

,year,num_nodes,num_edges,female_share_pct,assortativity,same_gender_share_obs_pct,same_gender_share_null_pct,same_gender_excess_pp
0,2011,34766,34782,27.763,0.122,63.990,59.935,4.055
1,2012,37500,40343,29.040,0.127,62.722,58.781,3.941
2,2013,38578,44395,29.291,0.116,61.903,58.507,3.397
3,2014,40691,47730,30.862,0.126,61.016,57.351,3.665
4,2015,43061,51830,31.667,0.131,60.976,56.715,4.261
5,2016,43143,53298,32.782,0.150,61.012,55.893,5.119
6,2017,45835,60384,32.936,0.135,60.317,55.782,4.535
7,2018,49302,67410,34.043,0.136,60.055,55.080,4.975
8,2019,52065,72425,35.179,0.135,59.682,54.403,5.279
9,2020,58746,84159,35.878,0.141,59.518,53.985,5.533


In [ ]:
data_flow = pd.DataFrame([
    ["authors rows", len(authors_15)],
    ["unique papers", authors_15["paper_id"].nunique()],
    ["unique authors", authors_15["author_id"].nunique()],
    ["pairs", len(pairs_15)],
    ["papers with pairs", pairs_15["paper_id"].nunique()],
    ["focal pairs", len(pairs_focal_15)],
    ["focal papers with pairs", pairs_focal_15["paper_id"].nunique()],
    ["edges", len(edges_15)],
    ["graph nodes", g_15.number_of_nodes()],
    ["graph edges", g_15.number_of_edges()],
    ["isolates", nx.number_of_isolates(g_15)],
    ["classified nodes", classified_graph(g_15).number_of_nodes()],
    ["classified edges", classified_graph(g_15).number_of_edges()]], columns=["step", "count"])

data_flow

,step,count
0,authors rows,1224074
1,unique papers,469010
2,unique authors,555415
3,pairs,1875891
4,papers with pairs,315386
5,focal pairs,1552322
6,focal papers with pairs,314833
7,edges,1204181
8,graph nodes,555415
9,graph edges,1204181


In [ ]:
cleaning_summary = pd.DataFrame([
    ["removed paratext papers", len(bad_papers)],
    ["author-paper rows in removed papers", bad_papers["num_authors"].sum()],
    ["possible false pairs", int(bad_papers["num_pairs"].sum())]], columns=["metric", "count"])

cleaning_summary

,metric,count
0,removed paratext papers,344
1,author-paper rows in removed papers,1352
2,possible false pairs,4038


In [ ]:
paper_sizes = (
    authors_15
    .groupby("paper_id")["author_id"]
    .nunique()
    .rename("num_authors"))

team_size_summary = paper_sizes.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99])

team_size_summary

,num_authors
count,469010.000000
mean,2.609910
std,1.948756
min,1.000000
25%,1.000000
50%,2.000000
75%,3.000000
90%,5.000000
95%,6.000000
99%,11.000000


In [ ]:
gender_coverage_country = (
    nodes_target
    .groupby("main_country_target")
    .agg(
        authors=("author_id", "nunique"),
        female=("gender", lambda x: (x == "female").sum()),
        male=("gender", lambda x: (x == "male").sum()),
        unknown=("gender", lambda x: (x == "unknown").sum())))

gender_coverage_country["classified"] = gender_coverage_country["female"] + gender_coverage_country["male"]
gender_coverage_country["classified_share"] = gender_coverage_country["classified"] / gender_coverage_country["authors"]
gender_coverage_country["unknown_share"] = gender_coverage_country["unknown"] / gender_coverage_country["authors"]

gender_coverage_country = gender_coverage_country.round(3)

gender_coverage_country

,authors,female,male,unknown,classified,classified_share,unknown_share
main_country_target,,,,,,,
br,35902,11884,19590,4428,31474,0.877,0.123
gb,64744,21454,34703,8587,56157,0.867,0.133
it,26434,9506,14883,2045,24389,0.923,0.077
kr,13465,1962,2841,8662,4803,0.357,0.643
ru,42545,13509,12359,16677,25868,0.608,0.392
us,230260,79002,119519,31739,198521,0.862,0.138


In [ ]:
foreign_coauthors = (
    nodes_15[nodes_15["main_country_target"].isna()]
    ["main_country_all"]
    .value_counts()
    .head(20)
    .reset_index())

foreign_coauthors.columns = ["main_country_all", "num_authors"]

foreign_coauthors

,main_country_all,num_authors
0,cn,17267
1,de,5753
2,ca,5137
3,fr,4324
4,au,4306
5,in,3714
6,nl,3626
7,es,3539
8,ch,2593
9,jp,1890


In [ ]:
final_data_overview = pd.DataFrame([
    ["period", "2011-2025"],
    ["target countries", "US, GB, IT, RU, BR, KR"],
    ["unique papers", authors_15["paper_id"].nunique()],
    ["unique authors", authors_15["author_id"].nunique()],
    ["focal pairs", len(pairs_focal_15)],
    ["edges", len(edges_15)],
    ["nodes", g_15.number_of_nodes()],
    ["isolates", nx.number_of_isolates(g_15)],
    ["classified nodes", classified_graph(g_15).number_of_nodes()],
    ["classified edges", classified_graph(g_15).number_of_edges()]], columns=["metric", "value"])

final_data_overview

,metric,value
0,period,2011-2025
1,target countries,"US, GB, IT, RU, BR, KR"
2,unique papers,469010
3,unique authors,555415
4,focal pairs,1552322
5,edges,1204181
6,nodes,555415
7,isolates,65470
8,classified nodes,417570
9,classified edges,838070


In [ ]:
left = edge_gender_15[[
    "author_1_id", "author_2_id", "country_1", "country_2", "weight"]].rename(columns={"country_1": "country", "country_2": "partner_country"})

right = edge_gender_15[[
    "author_1_id", "author_2_id", "country_2", "country_1", "weight"]].rename(columns={"country_2": "country", "country_1": "partner_country"})

focal_country_edges = pd.concat([left, right], ignore_index=True)
focal_country_edges = focal_country_edges[focal_country_edges["country"].notna()].copy()

focal_country_edges = focal_country_edges.drop_duplicates(
    subset=["author_1_id", "author_2_id", "country"])

focal_country_edges["scope"] = np.where(
    focal_country_edges["partner_country"] == focal_country_edges["country"],
    "domestic",
    "international")

collab_scope_country = pd.crosstab(
    focal_country_edges["country"],
    focal_country_edges["scope"],
    normalize="index").round(3)

collab_scope_country

scope,domestic,international
country,,
br,0.725,0.275
gb,0.457,0.543
it,0.568,0.432
kr,0.601,0.399
ru,0.709,0.291
us,0.703,0.297


In [ ]:
author_year = (authors_15.groupby(["publication_year", "author_id"], as_index=False).size().drop(columns="size"))

author_year = author_year.merge(nodes_15[["author_id", "main_country_target", "gender"]],on="author_id",how="left")

author_year_target = author_year[author_year["main_country_target"].notna()].copy()

gender_coverage_year_country = (author_year_target.groupby(["publication_year", "main_country_target"]).agg(
    authors=("author_id", "nunique"),
    female=("gender", lambda x: (x == "female").sum()),
    male=("gender", lambda x: (x == "male").sum()),
    unknown=("gender", lambda x: (x == "unknown").sum())).reset_index())

gender_coverage_year_country["classified"] = (gender_coverage_year_country["female"] + gender_coverage_year_country["male"])

gender_coverage_year_country["classified_share"] = (gender_coverage_year_country["classified"] / gender_coverage_year_country["authors"])

gender_coverage_year_country["unknown_share"] = (gender_coverage_year_country["unknown"] / gender_coverage_year_country["authors"])

gender_coverage_year_country = gender_coverage_year_country.round(3)

gender_coverage_year_country

,publication_year,main_country_target,authors,female,male,unknown,classified,classified_share,unknown_share
0,2011,br,1801,456,1122,223,1578,0.876,0.124
1,2011,gb,6434,1626,4212,596,5838,0.907,0.093
2,2011,it,2515,728,1693,94,2421,0.963,0.037
3,2011,kr,964,111,211,642,322,0.334,0.666
4,2011,ru,811,140,340,331,480,0.592,0.408
...,...,...,...,...,...,...,...,...,...
85,2025,gb,9168,3083,4948,1137,8031,0.876,0.124
86,2025,it,4410,1499,2696,215,4195,0.951,0.049
87,2025,kr,1866,286,423,1157,709,0.380,0.620
88,2025,ru,4048,1136,1273,1639,2409,0.595,0.405


In [ ]:
gender_coverage_year_country.to_csv("data/gender_coverage_year_country_2011_2025.csv",index=False)

SUBGRAPHS


In [ ]:
def graph_from_pairs(df, node_ids=None):
    edges = (
        df.groupby(["u", "v"], as_index=False)
        .agg(weight=("paper_id", "nunique"))
        .rename(columns={"u": "author_1_id", "v": "author_2_id"}))

    G = nx.Graph()

    for _, row in edges.iterrows():
        G.add_edge(
            row["author_1_id"],
            row["author_2_id"],
            weight=int(row["weight"]))

    if node_ids is not None:
        G.add_nodes_from(node_ids)

    nx.set_node_attributes(
        G, {n: node_attrs[n] for n in G.nodes() if n in node_attrs})

    return G

In [ ]:
def year_graph(year):
    df = pairs_focal_15[pairs_focal_15["publication_year"] == year].copy()
    node_ids = authors_15[authors_15["publication_year"] == year]["author_id"].unique()
    return graph_from_pairs(df, node_ids=node_ids)

In [ ]:
def country_subgraph(G, country):
    nodes = [
        n for n, d in G.nodes(data=True)
        if d.get("main_country_target") == country]
    return G.subgraph(nodes).copy()

In [ ]:
def classified_graph(G):
    nodes = [n for n, d in G.nodes(data=True)
        if d.get("gender") in ["female", "male"]]
    return G.subgraph(nodes).copy()

In [ ]:
g_2021 = year_graph(2021)
g_us = country_subgraph(g_15, "us")
g_us_2021 = country_subgraph(g_2021, "us")

print(g_2021.number_of_nodes(), g_2021.number_of_edges())
print(g_us.number_of_nodes(), g_us.number_of_edges())
print(g_us_2021.number_of_nodes(), g_us_2021.number_of_edges())

76478 120934
230260 543145
31673 52131


In [ ]:
g_classified_15 = classified_graph(g_15)

assort_15 = nx.attribute_assortativity_coefficient(g_classified_15, "gender")

print("classified nodes:", g_classified_15.number_of_nodes())
print("classified edges:", g_classified_15.number_of_edges())
print("gender assortativity:", assort_15)

classified nodes: 417570
classified edges: 838070
gender assortativity: 0.1402743023418537


In [ ]:
country_assort_rows = []

for country in sorted(nodes_15["main_country_target"].dropna().unique()):
    subg = country_subgraph(g_classified_15, country)

    if subg.number_of_edges() == 0:
        assort = np.nan
    else:
        assort = nx.attribute_assortativity_coefficient(subg, "gender")

    country_assort_rows.append({
        "country": country,
        "num_nodes": subg.number_of_nodes(),
        "num_edges": subg.number_of_edges(),
        "gender_assortativity": assort})

country_assort_15 = pd.DataFrame(country_assort_rows)

country_assort_15

,country,num_nodes,num_edges,gender_assortativity
0,br,31474,43352,0.160940
1,gb,56157,89140,0.126036
2,it,24389,45771,0.106266
3,kr,4803,2482,0.157055
4,ru,25868,19687,0.145843
5,us,198521,433216,0.144521


In [ ]:
country_assort_15.to_csv("data/country_gender_assortativity_2011_2025.csv", index=False)

In [ ]:
def gender_shuffle_null(G, B=100, seed=42):
    H = classified_graph(G)

    obs = nx.attribute_assortativity_coefficient(H, "gender")

    nodes = list(H.nodes())
    genders = np.array([H.nodes[n]["gender"] for n in nodes])

    rng = np.random.default_rng(seed)
    vals = []

    for _ in range(B):
        shuffled = rng.permutation(genders)
        nx.set_node_attributes(H, dict(zip(nodes, shuffled)), "gender_rand")
        vals.append(nx.attribute_assortativity_coefficient(H, "gender_rand"))

    vals = np.array(vals)

    return pd.Series({
        "observed": obs,
        "null_mean": vals.mean(),
        "null_sd": vals.std(ddof=1),
        "z_score": (obs - vals.mean()) / vals.std(ddof=1)})

In [ ]:
gender_shuffle_null(g_15, B=100, seed=42)

,0
observed,0.140274
null_mean,-0.000222
null_sd,0.000973
z_score,144.326277


In [ ]:
def random_edge_table(G, B=100, seed=42):
    H = classified_graph(G)

    edge_types = ["female-female", "female-male", "male-male"]

    obs_counts = {t: 0 for t in edge_types}

    for u, v in H.edges():
        t = edge_gender(H.nodes[u]["gender"], H.nodes[v]["gender"])
        obs_counts[t] += 1

    nodes = list(H.nodes())
    genders = np.array([H.nodes[n]["gender"] for n in nodes])

    rng = np.random.default_rng(seed)
    null_results = []

    for _ in range(B):
        shuffled = rng.permutation(genders)
        gender_rand = dict(zip(nodes, shuffled))

        rand_counts = {t: 0 for t in edge_types}

        for u, v in H.edges():
            t = edge_gender(gender_rand[u], gender_rand[v])
            rand_counts[t] += 1

        null_results.append(rand_counts)

    null_df = pd.DataFrame(null_results)

    result_df = pd.DataFrame({
        "edge_type": edge_types,
        "observed": [obs_counts[t] for t in edge_types],
        "random_mean": [null_df[t].mean() for t in edge_types],
        "random_sd": [null_df[t].std(ddof=1) for t in edge_types]})

    result_df["difference"] = result_df["observed"] - result_df["random_mean"]
    result_df["pct_diff"] = 100 * result_df["difference"] / result_df["random_mean"]
    result_df["z_score"] = result_df["difference"] / result_df["random_sd"]

    result_df["observed_share"] = result_df["observed"] / result_df["observed"].sum()
    result_df["random_share"] = result_df["random_mean"] / result_df["random_mean"].sum()
    result_df["share_diff_pp"] = 100 * (result_df["observed_share"] - result_df["random_share"])

    return result_df.round(2)

In [ ]:
edge_random_table_15 = random_edge_table(g_15, B=100, seed=42)

edge_random_table_15

,edge_type,observed,random_mean,random_sd,difference,pct_diff,z_score,observed_share,random_share,share_diff_pp
0,female-female,157326,130790.43,842.34,26535.57,20.29,31.50,0.19,0.16,3.17
1,female-male,343686,400681.17,567.50,-56995.17,-14.22,-100.43,0.41,0.48,-6.80
2,male-male,337058,306598.40,1247.45,30459.60,9.93,24.42,0.40,0.37,3.63


NULL MODEL WITH RANDOM EDGES

In [ ]:
edge_null_df = edges_15.copy()

lookup = nodes_15[["author_id", "gender", "main_country_all", "main_country_target"]].copy()

edge_null_df = edge_null_df.merge(
    lookup.rename(columns={
        "author_id": "author_1_id",
        "gender": "gender_1",
        "main_country_all": "country_1_all",
        "main_country_target": "country_1_target"}),
    on="author_1_id",
    how="left")

edge_null_df = edge_null_df.merge(
    lookup.rename(columns={
        "author_id": "author_2_id",
        "gender": "gender_2",
        "main_country_all": "country_2_all",
        "main_country_target": "country_2_target"}),
    on="author_2_id",
    how="left")

edge_null_df = edge_null_df[
    edge_null_df["gender_1"].isin(["female", "male"]) &
    edge_null_df["gender_2"].isin(["female", "male"]) &
    edge_null_df["country_1_all"].notna() &
    edge_null_df["country_2_all"].notna()].copy()

edge_null_df["c1"] = edge_null_df[["country_1_all", "country_2_all"]].min(axis=1)
edge_null_df["c2"] = edge_null_df[["country_1_all", "country_2_all"]].max(axis=1)

edge_null_df["left"] = np.where(
    edge_null_df["country_1_all"] == edge_null_df["c1"],
    edge_null_df["author_1_id"],
    edge_null_df["author_2_id"])

edge_null_df["right"] = np.where(
    edge_null_df["country_1_all"] == edge_null_df["c1"],
    edge_null_df["author_2_id"],
    edge_null_df["author_1_id"])

print("edges for null model:", len(edge_null_df))
print("country pairs:", edge_null_df[["c1", "c2"]].drop_duplicates().shape[0])

edges for null model: 838061
country pairs: 2544


In [ ]:
print("all edges:", len(edges_15))
print("classified edges with country_all:", len(edge_null_df))


all edges: 1204181
classified edges with country_all: 838061


In [ ]:
def rewire_within_country(edges, rng, swaps_per_edge=2):
    edges = [tuple(sorted(e)) for e in edges]

    if len(edges) < 2:
        return edges

    edge_set = set(edges)
    nswap = swaps_per_edge * len(edges)
    max_tries = 20 * nswap

    done = 0
    tries = 0

    while done < nswap and tries < max_tries:
        tries += 1

        i, j = rng.choice(len(edges), size=2, replace=False)

        a, b = edges[i]
        c, d = edges[j]

        if rng.random() < 0.5:
            a, b = b, a
        if rng.random() < 0.5:
            c, d = d, c

        new_1 = tuple(sorted((a, d)))
        new_2 = tuple(sorted((c, b)))

        old_1 = edges[i]
        old_2 = edges[j]

        if len(set(new_1)) < 2 or len(set(new_2)) < 2:
            continue
        if new_1 == new_2:
            continue
        if new_1 in edge_set and new_1 not in {old_1, old_2}:
            continue
        if new_2 in edge_set and new_2 not in {old_1, old_2}:
            continue

        edge_set.remove(old_1)
        edge_set.remove(old_2)
        edge_set.add(new_1)
        edge_set.add(new_2)

        edges[i] = new_1
        edges[j] = new_2

        done += 1

    return edges

In [ ]:
def rewire_between_countries(edges, rng, swaps_per_edge=2):
    edges = list(edges)

    if len(edges) < 2:
        return edges

    edge_set = set(edges)
    nswap = swaps_per_edge * len(edges)
    max_tries = 20 * nswap

    done = 0
    tries = 0

    while done < nswap and tries < max_tries:
        tries += 1

        i, j = rng.choice(len(edges), size=2, replace=False)

        a, b = edges[i]
        c, d = edges[j]

        new_1 = (a, d)
        new_2 = (c, b)

        old_1 = edges[i]
        old_2 = edges[j]

        if new_1 == new_2:
            continue
        if new_1 in edge_set and new_1 not in {old_1, old_2}:
            continue
        if new_2 in edge_set and new_2 not in {old_1, old_2}:
            continue

        edge_set.remove(old_1)
        edge_set.remove(old_2)
        edge_set.add(new_1)
        edge_set.add(new_2)

        edges[i] = new_1
        edges[j] = new_2

        done += 1

    return edges

In [ ]:
edge_groups = []

for (years, c1, c2), group in edge_null_df.groupby(["years", "c1", "c2"]):
    if c1 == c2:
        edges = list(zip(group["author_1_id"], group["author_2_id"]))
        edge_groups.append(("within", edges))
    else:
        edges = list(zip(group["left"], group["right"]))
        edge_groups.append(("between", edges))

print("number of edge groups:", len(edge_groups))

number of edge groups: 31923


In [ ]:
gender_lookup = nodes_15.set_index("author_id")["gender"].to_dict()

edge_types = ["female-female", "female-male", "male-male"]

def count_gender_edges(edges):
    counts = {t: 0 for t in edge_types}

    for u, v in edges:
        t = edge_gender(gender_lookup[u], gender_lookup[v])

        if t in counts:
            counts[t] += 1

    return counts

In [ ]:
def rewired_edge_null_table(B=50, seed=42, swaps_per_edge=2):
    rng = np.random.default_rng(seed)

    observed_edges = list(zip(edge_null_df["author_1_id"], edge_null_df["author_2_id"]))
    obs_counts = count_gender_edges(observed_edges)

    null_results = []

    for b in range(B):
        random_edges = []

        for kind, edges in edge_groups:
            if kind == "within":
                new_edges = rewire_within_country(edges, rng, swaps_per_edge=swaps_per_edge)
            else:
                new_edges = rewire_between_countries(edges, rng, swaps_per_edge=swaps_per_edge)

            random_edges.extend(new_edges)

        null_results.append(count_gender_edges(random_edges))

        if (b + 1) % 10 == 0:
            print("done:", b + 1, "/", B)

    null_df = pd.DataFrame(null_results)

    result_df = pd.DataFrame({
        "edge_type": edge_types,
        "observed": [obs_counts[t] for t in edge_types],
        "random_mean": [null_df[t].mean() for t in edge_types],
        "random_sd": [null_df[t].std(ddof=1) for t in edge_types]})

    result_df["difference"] = result_df["observed"] - result_df["random_mean"]
    result_df["pct_diff"] = 100 * result_df["difference"] / result_df["random_mean"]
    result_df["z_score"] = result_df["difference"] / result_df["random_sd"]

    result_df["observed_share"] = result_df["observed"] / result_df["observed"].sum()
    result_df["random_share"] = result_df["random_mean"] / result_df["random_mean"].sum()
    result_df["share_diff_pp"] = 100 * (result_df["observed_share"] - result_df["random_share"])

    return result_df.round(2)

In [ ]:
rewired_table_15 = rewired_edge_null_table(B=20, seed=42, swaps_per_edge=2)

rewired_table_15

done: 10 / 20
done: 20 / 20


,edge_type,observed,random_mean,random_sd,difference,pct_diff,z_score,observed_share,random_share,share_diff_pp
0,female-female,157324,135513.7,193.93,21810.3,16.09,112.47,0.19,0.16,2.6
1,female-male,343684,387304.6,387.86,-43620.6,-11.26,-112.47,0.41,0.46,-5.2
2,male-male,337053,315242.7,193.93,21810.3,6.92,112.47,0.40,0.38,2.6


In [ ]:
def compact_rewired_summary(tbl):
    d = tbl.set_index("edge_type")

    ff_obs = d.loc["female-female", "observed"]
    fm_obs = d.loc["female-male", "observed"]
    mm_obs = d.loc["male-male", "observed"]

    ff_rand = d.loc["female-female", "random_mean"]
    fm_rand = d.loc["female-male", "random_mean"]
    mm_rand = d.loc["male-male", "random_mean"]

    total_obs = ff_obs + fm_obs + mm_obs
    total_rand = ff_rand + fm_rand + mm_rand

    rows = [
        ["same-gender edges", ff_obs + mm_obs, ff_rand + mm_rand],
        ["mixed-gender edges", fm_obs, fm_rand],
        ["female-female share", ff_obs / total_obs, ff_rand / total_rand],
        ["female-male share", fm_obs / total_obs, fm_rand / total_rand],
        ["male-male share", mm_obs / total_obs, mm_rand / total_rand],
        ["same-gender share", (ff_obs + mm_obs) / total_obs, (ff_rand + mm_rand) / total_rand]]

    out = pd.DataFrame(rows, columns=["metric", "observed", "random_mean"])

    out["difference"] = out["observed"] - out["random_mean"]
    out["pct_diff"] = 100 * out["difference"] / out["random_mean"]
    out["difference_pp"] = np.where(out["metric"].str.contains("share"), 100 * out["difference"], np.nan)

    return out.round(4)

In [ ]:
rewired_summary_15 = compact_rewired_summary(rewired_table_15)

rewired_summary_15

,metric,observed,random_mean,difference,pct_diff,difference_pp
0,same-gender edges,494377.0000,450756.4000,43620.600,9.6772,NaN
1,mixed-gender edges,343684.0000,387304.6000,-43620.600,-11.2626,NaN
2,female-female share,0.1877,0.1617,0.026,16.0945,2.6025
3,female-male share,0.4101,0.4621,-0.052,-11.2626,-5.2049
4,male-male share,0.4022,0.3762,0.026,6.9186,2.6025
5,same-gender share,0.5899,0.5379,0.052,9.6772,5.2049


In [ ]:
num_components = nx.number_connected_components(g_15)
component_sizes = sorted([len(c) for c in nx.connected_components(g_15)], reverse=True)

giant_summary = pd.DataFrame({
    "metric": ["nodes","edges","connected_components","giant_component_size","giant_component_share","isolates","isolate_share"],
    "value": [g_15.number_of_nodes(),g_15.number_of_edges(),num_components,component_sizes[0],
              component_sizes[0] / g_15.number_of_nodes(),nx.number_of_isolates(g_15),nx.number_of_isolates(g_15) / g_15.number_of_nodes()]})

giant_summary

,metric,value
0,nodes,5.554150e+05
1,edges,1.204181e+06
2,connected_components,1.152090e+05
3,giant_component_size,3.093790e+05
4,giant_component_share,5.570231e-01
5,isolates,6.547000e+04
6,isolate_share,1.178758e-01


In [ ]:
giant_summary_show = giant_summary.copy()

giant_summary_show["value"] = giant_summary_show["value"].apply(lambda x: round(x, 3) if x < 1 else int(x))

giant_summary_show

,metric,value
0,nodes,555415.000
1,edges,1204181.000
2,connected_components,115209.000
3,giant_component_size,309379.000
4,giant_component_share,0.557
5,isolates,65470.000
6,isolate_share,0.118


In [ ]:
country_components = []

for country in sorted(nodes_target["main_country_target"].unique()):
    H = country_subgraph(g_15, country)
    sizes = sorted([len(c) for c in nx.connected_components(H)], reverse=True)

    country_components.append({
        "country": country,
        "nodes": H.number_of_nodes(),
        "edges": H.number_of_edges(),
        "connected_components": nx.number_connected_components(H),
        "giant_component_size": sizes[0],
        "giant_component_share": sizes[0] / H.number_of_nodes(),
        "isolates": nx.number_of_isolates(H),
        "isolate_share": nx.number_of_isolates(H) / H.number_of_nodes()})

country_components = pd.DataFrame(country_components).round(3)

country_components

,country,nodes,edges,connected_components,giant_component_size,giant_component_share,isolates,isolate_share
0,br,35902,56658,11044,13191,0.367,6401,0.178
1,gb,64744,110859,24525,28027,0.433,19136,0.296
2,it,26434,53027,8171,12778,0.483,6041,0.229
3,kr,13465,20698,4164,5425,0.403,2589,0.192
4,ru,42545,42987,18851,7910,0.186,11880,0.279
5,us,230260,543145,66860,122006,0.530,49154,0.213


In [ ]:
edge_random_table_15.to_csv("data/edge_random_table_2011_2025_clean.csv", index=False)
rewired_table_15.to_csv("data/rewired_table_2011_2025_clean.csv", index=False)
rewired_summary_15.to_csv("data/rewired_summary_2011_2025_clean.csv", index=False)

In [ ]:
print("FINAL CHECKS")

print("authors_15 rows:", len(authors_15))
print("unique papers:", authors_15["paper_id"].nunique())
print("unique authors:", authors_15["author_id"].nunique())

print("pairs_15 rows:", len(pairs_15))
print("pairs_focal_15 rows:", len(pairs_focal_15))

print("edges_15 rows:", len(edges_15))
print("g_15 nodes:", g_15.number_of_nodes())
print("g_15 edges:", g_15.number_of_edges())

print("edge rows == graph edges:", len(edges_15) == g_15.number_of_edges())

classified_edges_graph = classified_graph(g_15).number_of_edges()
classified_edges_table = (edge_gender_15["edge_gender_type"] != "unknown").sum()

print("classified nodes:", classified_graph(g_15).number_of_nodes())
print("classified edges in graph:", classified_edges_graph)
print("classified edges in edge table:", classified_edges_table)
print("classified edges match:", classified_edges_graph == classified_edges_table)

print("isolates:", nx.number_of_isolates(g_15))
print("connected components:", nx.number_connected_components(g_15))

FINAL CHECKS
authors_15 rows: 1224074
unique papers: 469010
unique authors: 555415
pairs_15 rows: 1875891
pairs_focal_15 rows: 1552322
edges_15 rows: 1204181
g_15 nodes: 555415
g_15 edges: 1204181
edge rows == graph edges: True
classified nodes: 417570
classified edges in graph: 838070
classified edges in edge table: 838070
classified edges match: True
isolates: 65470
connected components: 115209


In [ ]:
top_papers = (authors_15.groupby(["paper_id", "paper_title", "publication_year"])["author_id"]
              .nunique().reset_index(name="num_authors").sort_values("num_authors", ascending=False))

top_papers.head(30)

,paper_id,paper_title,publication_year,num_authors
338952,https://openalex.org/W4281667173,"Trainee Perspectives on Race, Antiracism, and ...",2022,15
142083,https://openalex.org/W2606276971,35 innovators under 35 2013,2013,15
196598,https://openalex.org/W2945686055,The social and economic toll of cancer survivo...,2019,15
142046,https://openalex.org/W2606189110,The development of China’s Yangtze River Econo...,2017,15
312185,https://openalex.org/W3215800711,Protecting the poor with a carbon tax and equa...,2021,15
167137,https://openalex.org/W2793507334,Antitrust Law and Policies from Brussels to Wa...,2014,15
86077,https://openalex.org/W2176598210,Can Paris pledges avert severe climate change?,2015,15
406505,https://openalex.org/W4394598570,Feasibility of Using Blood-based Biomarkers to...,2024,15
138261,https://openalex.org/W2593395519,Recommendations from the European Working Grou...,2017,15
323019,https://openalex.org/W4220781979,Consolidated Health Economic Evaluation Report...,2022,15
